# Gold booking revenue, daily
Reference output for Prompt 2b-1 once the instructions and skills are on (`de-job-standards`, `table-governance`, `data-quality-checks`).

Writes `{catalog}.{schema}.gold_booking_revenue_daily`: one row per `booking_date`, `destination_id` and `property_type`, from `samples.wanderbricks.bookings` joined to `properties` and `destinations`.

* **Net revenue excludes cancelled bookings** (`status = 'cancelled'`); gross and cancelled amounts are kept as separate columns.
* Idempotent: a full run rebuilds the table; a run with `start_date` replaces only booking dates from that day on (`replaceWhere`).
* Task of the `wanderbricks_booking_revenue_daily` job (tag `project:asml_demo`).

In [ ]:
# Target: {catalog}.{schema}.gold_booking_revenue_daily (PK: booking_date, destination_id, property_type)
from datetime import date

dbutils.widgets.text("catalog", "genie_code_workshop")
dbutils.widgets.text("schema", "asml_demo")
dbutils.widgets.text("start_date", "")  # empty = full rebuild; YYYY-MM-DD = reprocess from that booking date

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
start_date = dbutils.widgets.get("start_date").strip()
if start_date:
    date.fromisoformat(start_date)  # validate before using it in replaceWhere

target = f"{catalog}.{schema}.gold_booking_revenue_daily"
print(target, start_date or "(full rebuild)")

In [ ]:
from pyspark.sql import functions as F

bookings = spark.table("samples.wanderbricks.bookings").withColumn("booking_date", F.to_date("created_at"))
if start_date:
    bookings = bookings.filter(F.col("booking_date") >= F.lit(start_date).cast("date"))

properties = spark.table("samples.wanderbricks.properties").select("property_id", "destination_id", "property_type")
destinations = spark.table("samples.wanderbricks.destinations").select(
    "destination_id", "destination", F.col("country").alias("destination_country")
)

is_cancelled = F.col("status") == "cancelled"

df = (
    bookings.join(properties, "property_id", "inner")
    .join(destinations, "destination_id", "inner")
    .groupBy("booking_date", "destination_id", "destination", "destination_country", "property_type")
    .agg(
        F.count("*").alias("booking_count"),
        F.sum(F.when(is_cancelled, 1).otherwise(0)).alias("cancelled_booking_count"),
        F.sum(F.when(~is_cancelled, F.datediff("check_out", "check_in")).otherwise(0)).alias("booked_nights"),
        F.round(F.sum("total_amount"), 2).alias("gross_booking_amount"),
        F.round(F.sum(F.when(is_cancelled, F.col("total_amount")).otherwise(0)), 2).alias("cancelled_amount"),
        F.round(F.sum(F.when(~is_cancelled, F.col("total_amount")).otherwise(0)), 2).alias("net_revenue"),
    )
    .withColumn("net_booking_count", F.col("booking_count") - F.col("cancelled_booking_count"))
    .withColumn(
        "avg_net_booking_value",
        F.round(F.col("net_revenue") / F.nullif(F.col("net_booking_count"), F.lit(0)), 2),
    )
    .withColumn("audit_timestamp", F.current_timestamp())
    .withColumn("source_system", F.lit("samples.wanderbricks"))
    # keys -> dimensions -> measures -> derived -> audit columns LAST
    .select(
        "booking_date", "destination_id", "property_type",
        "destination", "destination_country",
        "booking_count", "cancelled_booking_count", "net_booking_count", "booked_nights",
        "gross_booking_amount", "cancelled_amount", "net_revenue", "avg_net_booking_value",
        "audit_timestamp", "source_system",
    )
)

if df.limit(1).count() == 0:
    raise ValueError(f"No rows produced for {target}")

In [ ]:
# Full rebuild, or replace only booking dates >= start_date (re-runs never duplicate rows)
writer = df.write.format("delta")
if start_date and spark.catalog.tableExists(target):
    writer.mode("overwrite").option("replaceWhere", f"booking_date >= '{start_date}'").saveAsTable(target)
else:
    writer.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)

print(f"{spark.table(target).count()} rows in {target}")

In [ ]:
# DDL per table-governance standard; re-applied on every run because overwriteSchema resets comments
spark.sql(f"""COMMENT ON TABLE {target} IS
  'Business aggregation: daily booking revenue by booking date, destination and property type from samples.wanderbricks.bookings. Net revenue excludes cancelled bookings.'""")

spark.sql(f"""ALTER TABLE {target} SET TBLPROPERTIES (
  'quality' = 'gold',
  'domain' = 'bookings',
  'contains_pii' = 'false',
  'delta.enableChangeDataFeed' = 'true')""")

column_comments = {
    "booking_date": "Date the booking was created (to_date(bookings.created_at)). Part of the primary key",
    "destination_id": "Part of the primary key. References samples.wanderbricks.destinations.destination_id",
    "property_type": "Part of the primary key. Property type from samples.wanderbricks.properties",
    "destination": "Destination name, from samples.wanderbricks.destinations",
    "destination_country": "Destination country, from samples.wanderbricks.destinations.country",
    "booking_count": "COUNT of all bookings, any status",
    "cancelled_booking_count": "COUNT of bookings with status = cancelled",
    "net_booking_count": "Calculated as booking_count - cancelled_booking_count",
    "booked_nights": "SUM(check_out - check_in) in nights over non-cancelled bookings",
    "gross_booking_amount": "SUM(total_amount) over all bookings, any status. Units: currency of total_amount",
    "cancelled_amount": "SUM(total_amount) over cancelled bookings. Units: currency of total_amount",
    "net_revenue": "Calculated as SUM(total_amount) excluding cancelled bookings. Units: currency of total_amount",
    "avg_net_booking_value": "Calculated as net_revenue / net_booking_count, null when there are no net bookings",
    "audit_timestamp": "Job execution timestamp",
    "source_system": "Upstream source system identifier",
}
for col, comment in column_comments.items():
    spark.sql(f"ALTER TABLE {target} ALTER COLUMN {col} COMMENT '{comment}'")

# Governed tag policies in this account reject 'domain' (no allowed values),
# so the domain is recorded in TBLPROPERTIES instead
spark.sql(f"""ALTER TABLE {target} SET TAGS (
  'quality' = 'gold', 'data_classification' = 'internal', 'sla' = 'daily')""")

In [ ]:
# Copy of asml-workshop/skills/job-standards/scripts/check_table_standards.py (metadata reads only)
import re

LAYER_PREFIXES = ("bronze_", "silver_", "gold_")
REQUIRED_PROPS = ("quality", "domain")
AUDIT_COLUMNS = ["audit_timestamp", "source_system"]
NAME_PATTERN = re.compile(r"^[a-z][a-z0-9_]*$")

def check_table_standards(full_name: str) -> list:
    failures = []
    table = full_name.split(".")[2]
    if not NAME_PATTERN.match(table):
        failures.append(f"Table name '{table}' is not lowercase_snake_case")
    if not table.startswith(LAYER_PREFIXES):
        failures.append(f"Table name '{table}' has no layer prefix {LAYER_PREFIXES}")
    if not spark.sql(f"DESCRIBE DETAIL {full_name}").first()["description"]:
        failures.append("Missing table COMMENT")
    props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {full_name}").collect()}
    failures += [f"Missing TBLPROPERTY '{p}'" for p in REQUIRED_PROPS if p not in props]
    if props.get("contains_pii") == "true" and not props.get("pii_columns"):
        failures.append("contains_pii=true but 'pii_columns' is not set")
    fields = spark.table(full_name).schema.fields
    if [f.name for f in fields][-2:] != AUDIT_COLUMNS:
        failures.append(f"Last two columns must be {AUDIT_COLUMNS}")
    undocumented = [f.name for f in fields if not f.metadata.get("comment")]
    if undocumented:
        failures.append(f"Columns without COMMENT: {undocumented}")
    print(f"[{'PASS' if not failures else f'FAIL ({len(failures)})'}] {full_name}")
    for f in failures:
        print(f"  - {f}")
    return failures

failures = check_table_standards(target)

t = spark.table(target)
# Primary key must be unique
dupes = t.groupBy("booking_date", "destination_id", "property_type").count().filter("count > 1").count()
if dupes:
    failures.append(f"{dupes} duplicate (booking_date, destination_id, property_type) rows")
# Net + cancelled must add up to gross (within rounding)
mismatch = t.filter(F.abs(F.col("net_revenue") + F.col("cancelled_amount") - F.col("gross_booking_amount")) > 0.05).count()
if mismatch:
    failures.append(f"{mismatch} rows where net_revenue + cancelled_amount != gross_booking_amount")
# Reconcile with the source: net revenue total must match bookings excluding cancelled
if not start_date:
    src = spark.sql("SELECT round(sum(total_amount), 0) AS v FROM samples.wanderbricks.bookings WHERE status <> 'cancelled'").first()["v"]
    tgt = t.agg(F.round(F.sum("net_revenue"), 0).alias("v")).first()["v"]
    if abs(src - tgt) > 1:
        failures.append(f"net_revenue total {tgt} does not reconcile with source {src}")

if failures:
    raise AssertionError(f"{target} does not meet standards: {failures}")

display(t.groupBy("property_type").agg(F.sum("net_revenue").alias("net_revenue"), F.sum("cancelled_amount").alias("cancelled_amount")).orderBy(F.desc("net_revenue")))